# 벡터 데이터베이스와 검색 (Vector Search with ChromaDB)

**Skilljar Lessons L04-L05 대응**

이 노트북에서 다루는 내용:
1. ChromaDB 벡터 데이터베이스 설정
2. VectorIndex 클래스 구현
3. 전체 RAG 파이프라인 (인덱싱 → 검색 → 생성)
4. Claude와 연동한 RAG 질의응답

In [ ]:
# ── Setup ──────────────────────────────────────────────
# pip install chromadb voyageai anthropic python-dotenv

import anthropic
import chromadb
import voyageai
import numpy as np
import re
from dotenv import load_dotenv

load_dotenv()

claude_client = anthropic.Anthropic()
voyage_client = voyageai.Client()
MODEL = "claude-haiku-4-5"

## §1. 청킹 함수 (S4_01에서 가져옴)

In [ ]:
def chunk_by_char(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    """텍스트를 고정 문자 수로 분할한다."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

## §2. VectorIndex 클래스

ChromaDB와 VoyageAI를 통합한 벡터 검색 인덱스입니다.  
문서 추가 (`add_documents`)와 검색 (`search`)을 캡슐화합니다.

In [ ]:
class VectorIndex:
    """ChromaDB 기반 벡터 검색 인덱스"""

    def __init__(
        self,
        collection_name: str = "documents",
        embedding_model: str = "voyage-3"
    ):
        self.voyage_client = voyageai.Client()
        self.embedding_model = embedding_model

        self.chroma_client = chromadb.Client()
        self.collection = self.chroma_client.create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )

    def add_documents(self, documents: list[str], ids: list[str] = None):
        """문서 청크를 인덱스에 추가한다."""
        if ids is None:
            ids = [f"doc_{i}" for i in range(len(documents))]

        result = self.voyage_client.embed(
            texts=documents,
            model=self.embedding_model,
            input_type="document"
        )

        self.collection.add(
            documents=documents,
            embeddings=result.embeddings,
            ids=ids
        )
        print(f"\u2705 {len(documents)}개 문서 인덱싱 완료")

    def search(self, query: str, top_k: int = 5) -> list[dict]:
        """쿼리와 유사한 문서를 검색한다."""
        query_result = self.voyage_client.embed(
            texts=[query],
            model=self.embedding_model,
            input_type="query"
        )

        results = self.collection.query(
            query_embeddings=query_result.embeddings,
            n_results=top_k
        )

        search_results = []
        for i in range(len(results["documents"][0])):
            search_results.append({
                "text": results["documents"][0][i],
                "score": 1 - results["distances"][0][i],
                "id": results["ids"][0][i]
            })

        return search_results

## §3. 문서 인덱싱

샘플 문서를 청킹하고 VectorIndex에 인덱싱합니다.

In [ ]:
SAMPLE_DOCUMENT = """
콘크리트의 설계기준강도 fck는 최소 21 MPa 이상이어야 한다. 고강도 콘크리트의 경우 fck 40 MPa 이상을 적용할 수 있다. 철근의 항복강도 fy는 400 MPa 또는 500 MPa를 표준으로 한다.

RC 보의 최소 철근비는 0.25*sqrt(fck)/fy 이상이어야 하며, 1.4/fy 이상이어야 한다. 최대 철근비는 균형 철근비의 0.75배를 초과할 수 없다. 전단보강은 스터럽 간격이 d/2 이하가 되도록 배치해야 한다.

기둥의 최소 단면치수는 300mm 이상이어야 한다. 주근의 최소 개수는 4개이며, 최소 철근비는 0.01 이상이어야 한다. 띠철근의 간격은 주근 직경의 16배, 띠철근 직경의 48배, 기둥 최소 치수 중 가장 작은 값 이하로 한다.

고정하중은 구조물 자체의 무게와 영구적으로 부착된 부분의 무게를 포함한다. 콘크리트의 단위중량은 24 kN/m3, 철근콘크리트는 25 kN/m3을 표준값으로 한다.

적재하중은 건축물의 용도에 따라 다르게 적용한다. 주거용 건물의 바닥 적재하중은 2.0 kN/m2, 사무실은 2.5 kN/m2, 상점은 4.0 kN/m2를 적용한다.
""".strip()

# 청킹
chunks = chunk_by_char(SAMPLE_DOCUMENT, chunk_size=200, overlap=40)
print(f"{len(chunks)}개 청크 생성")

# 인덱싱
index = VectorIndex(collection_name="kds_sample")
index.add_documents(chunks)

## §4. 벡터 검색 테스트

In [ ]:
# 검색 테스트
queries = [
    "RC 보의 최소 철근비는?",
    "기둥의 최소 단면치수",
    "사무실 적재하중",
]

for query in queries:
    results = index.search(query, top_k=2)
    print(f"\n쿼리: '{query}'")
    for i, r in enumerate(results, 1):
        print(f"  {i}위 [{r['score']:.4f}] {r['text'][:80]}...")

## §5. Claude와 연동한 RAG 질의응답

검색된 청크를 컨텍스트로 Claude에게 전달하여 **근거 기반 답변**을 생성합니다.

In [ ]:
def rag_query(question: str, index: VectorIndex, top_k: int = 3) -> str:
    """RAG 기반 질의응답을 수행한다."""
    # 관련 청크 검색
    results = index.search(question, top_k=top_k)
    context = "\n\n---\n\n".join([r["text"] for r in results])

    print(f"검색된 청크 {len(results)}개:")
    for r in results:
        print(f"  [{r['score']:.4f}] {r['text'][:60]}...")
    print()

    # Claude에게 전달
    response = claude_client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=(
            "다음 문서를 참고하여 질문에 답하세요. "
            "문서에 없는 정보는 '문서에서 확인할 수 없습니다'라고 답하세요.\n\n"
            f"참고 문서:\n{context}"
        ),
        messages=[{"role": "user", "content": question}]
    )
    return response.content[0].text

In [ ]:
# RAG 질의응답 테스트
answer = rag_query("RC 보의 최소 철근비 기준을 설명해줘", index)
print(f"답변:\n{answer}")

In [ ]:
# 문서에 없는 내용 질문
answer = rag_query("프리스트레스트 콘크리트의 설계 기준은?", index)
print(f"답변:\n{answer}")

## 정리

- ChromaDB로 벡터 데이터베이스를 쉽게 구축할 수 있다
- VectorIndex 클래스로 인덱싱과 검색을 캡슐화
- 검색된 컨텍스트를 Claude의 system 프롬프트에 추가하여 RAG 구현
- "문서에 없는 정보는 답하지 마라"로 환각 방지

다음 노트북에서는 **BM25 키워드 검색**을 추가합니다. → `S4_04_bm25.ipynb`